# 🎵 Last.fm API Exploration — EDM Tags & Track Metadata

This notebook explores the Last.fm REST API (`/2.0/`) to evaluate crowdsourced user tag coverage, track/artist metadata availability, and MusicBrainz ID (MBID) link resolution for electronic music (EDM) sub-genre classification.

### Objectives
* Test connection authentication using `LASTFM_API_KEY` and custom `User-Agent` headers.
* Query top-performing tracks for target genre tags (`tag=edm`).
* Inspect raw JSON payload structure and nested key representations (`tracks -> track`).
* Assess MBID cross-referencing capabilities with MusicBrainz entities.

In [1]:
import os
import pandas as pd
from src.agies.integration.api_helpers import fetch_raw_api_sample
from data_ravers_utils import eda_utils

In [2]:
LASTFM_API_KEY = os.getenv("LASTFM_API_KEY")
LASTFM_BASE_URL = "http://ws.audioscrobbler.com/2.0/"
HEADERS = {"User-Agent": "AGIES/0.1 (info@dataravers.space)"}

## 1. API Request Configuration & Raw Sample Fetch

Execute a test request against the `tag.gettoptracks` method to retrieve raw JSON records for the target tag (`edm`).

In [3]:
# Fetch raw sample using your shared helper
params = {
    "method": "tag.gettoptracks",
    "tag": "edm",
    "api_key": LASTFM_API_KEY,
    "format": "json",
    "limit": 5
}

raw_data = fetch_raw_api_sample(
    url=LASTFM_BASE_URL, 
    params=params, 
    headers=HEADERS, 
    timeout=15
)

# Inspect the raw JSON response structure
raw_data

{'tracks': {'track': [{'name': 'Faded',
    'duration': '212',
    'mbid': '021eeff6-4680-46dc-bf5b-3271b3a713f5',
    'url': 'https://www.last.fm/music/Alan+Walker/_/Faded',
    'streamable': {'#text': '0', 'fulltrack': '0'},
    'artist': {'name': 'Alan Walker',
     'mbid': '71d524e4-2ae2-4a3e-baa0-0c021925db7a',
     'url': 'https://www.last.fm/music/Alan+Walker'},
    'image': [{'#text': 'https://lastfm-img.freetls.fastly.net/i/u/34s/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'small'},
     {'#text': 'https://lastfm-img.freetls.fastly.net/i/u/64s/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'medium'},
     {'#text': 'https://lastfm-img.freetls.fastly.net/i/u/174s/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'large'},
     {'#text': 'https://lastfm-img.freetls.fastly.net/i/u/300x300/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'extralarge'}],
    '@attr': {'rank': '1'}},
   {'name': 'Solo (feat. Demi Lovato)',
    'duration': '222',
    'mbid': '0a

## 2. Inspect Raw JSON Structure & Nested Payload Keys

Verify the root response schema and isolate the nested track dictionary list (`raw_data['tracks']['track']`).

In [4]:
records_list = raw_data.get("tracks",{}).get("track",[])
print(f"Total items found:{len(records_list)}")
records_list[0]

Total items found:5


{'name': 'Faded',
 'duration': '212',
 'mbid': '021eeff6-4680-46dc-bf5b-3271b3a713f5',
 'url': 'https://www.last.fm/music/Alan+Walker/_/Faded',
 'streamable': {'#text': '0', 'fulltrack': '0'},
 'artist': {'name': 'Alan Walker',
  'mbid': '71d524e4-2ae2-4a3e-baa0-0c021925db7a',
  'url': 'https://www.last.fm/music/Alan+Walker'},
 'image': [{'#text': 'https://lastfm-img.freetls.fastly.net/i/u/34s/2a96cbd8b46e442fc41c2b86b821562f.png',
   'size': 'small'},
  {'#text': 'https://lastfm-img.freetls.fastly.net/i/u/64s/2a96cbd8b46e442fc41c2b86b821562f.png',
   'size': 'medium'},
  {'#text': 'https://lastfm-img.freetls.fastly.net/i/u/174s/2a96cbd8b46e442fc41c2b86b821562f.png',
   'size': 'large'},
  {'#text': 'https://lastfm-img.freetls.fastly.net/i/u/300x300/2a96cbd8b46e442fc41c2b86b821562f.png',
   'size': 'extralarge'}],
 '@attr': {'rank': '1'}}

## 3. Tabular Representation & Metadata Evaluation

Convert the dictionary array directly into a DataFrame to confirm presence of core fields (`name`, `artist`, `mbid`, `url`, `@attr`).

In [5]:
df=pd.DataFrame(records_list)
print(df.head())

                         name duration                                  mbid  \
0                       Faded      212  021eeff6-4680-46dc-bf5b-3271b3a713f5   
1    Solo (feat. Demi Lovato)      222  0a88e8af-cf68-4f86-9466-0be628cc5d52   
2  Waste It On Me (feat. BTS)      192  5e5b2f0e-c5bb-4b6b-a1c0-96748b9b9ca3   
3                     Shelter      218                                   NaN   
4                  The Middle      184  117a0899-8ebc-47d4-882e-a00e4370d587   

                                                 url  \
0      https://www.last.fm/music/Alan+Walker/_/Faded   
1  https://www.last.fm/music/Clean+Bandit/_/Solo+...   
2  https://www.last.fm/music/Steve+Aoki/_/Waste+I...   
3  https://www.last.fm/music/Porter+Robinson/_/Sh...   
4        https://www.last.fm/music/Zedd/_/The+Middle   

                         streamable  \
0  {'#text': '0', 'fulltrack': '0'}   
1  {'#text': '0', 'fulltrack': '0'}   
2  {'#text': '0', 'fulltrack': '0'}   
3  {'#text': '0', 'fulltra